# Week 10 — Block 2: Guided Demo (Maps II)

**DATS 6401 · Visualization of Complex Data**

Week 9 built maps you look at. This week builds maps you can **ask questions
of**, and does the spatial work needed to answer them.

Running example: **2,264 Tesla Supercharger sites**, with coordinates, stall
counts and opening dates.

1. The same map three ways: folium, plotly, Altair
2. Point in polygon — the spatial join, and the three sites it drops
3. Sites and stalls per state, then per million people
4. The Albers USA layout, and what the real coordinates look like
5. Altair's selection: one view driving another
6. Distance: haversine, and what the nearest-neighbor column reveals
7. DBSCAN, and the metric mistake that quietly changes the answer
8. Heat maps, and a 3D extrusion with pydeck

**The maps need internet** — folium, plotly, Altair and pydeck all fetch their
JavaScript from a CDN. The analysis itself is entirely offline.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

sites = pd.read_csv("data/superchargers_us.csv")
states = gpd.read_file("data/states.geojson")
print(f"{len(sites)} sites, {sites.stalls.sum():,} stalls, {len(states)} state polygons")
sites.head(3)

## Part 1 — The same map, three ways

Each of these draws the sites. What differs is what the reader can then do.

In [ ]:
import folium

conus = sites[(sites.lat.between(24, 50)) & (sites.lon.between(-125, -66))].copy()

m = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap",
               scrollWheelZoom=False)
# One GeoJson layer, not one marker per point. Measured: 2,254 points are
# about 1,481 KB as CircleMarkers and about 401 KB this way.
folium.GeoJson(
    {"type": "FeatureCollection", "features": [
        {"type": "Feature",
         "geometry": {"type": "Point", "coordinates": [r.lon, r.lat]},
         "properties": {"name": r["name"], "stalls": int(r.stalls)}}
        for _, r in conus.iterrows()]},
    name="sites",
    marker=folium.CircleMarker(radius=3, fill=True, fill_opacity=0.7,
                               color="#d9534f", fill_color="#d9534f"),
    tooltip=folium.GeoJsonTooltip(fields=["name", "stalls"], aliases=["", "stalls:"]),
).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

**folium gives you a normal web map**: tiles you can pan and zoom, a tooltip on
hover, a layer control to switch things off. It is Leaflet, which is what most
of the web uses.

Hovering is the part that matters. The reader gets the number beside the point
instead of having to measure a color against a legend — and measuring color is
the thing people are worst at.

## Part 2 — Point in polygon

We want a choropleth of sites per state. The file *has* a `state` column, so we
could group on it.

**Do not.** That column is something a person typed. "Which state is this point
in" is a geometric question, and answering it geometrically is the skill —
because usually there is no such column.

In [ ]:
states_j = states[["fips", "state", "geometry"]].rename(columns={"state": "state_polygon"})
points = gpd.GeoDataFrame(sites, crs="EPSG:4326",
                          geometry=gpd.points_from_xy(sites.lon, sites.lat))

# how="left" keeps the points that match nothing. The default drops them.
tagged = gpd.sjoin(points, states_j, predicate="within", how="left")
print(f"{len(points)} points in, {tagged.fips.notna().sum()} matched a state, "
      f"{tagged.fips.isna().sum()} matched nothing")

Three matched nothing. Look at them before explaining them away.

In [ ]:
print(tagged[tagged.fips.isna()][["name", "city", "state", "lat", "lon"]].to_string(index=False))

**None of them is in the sea.** Vesey Street is in lower Manhattan, McDermitt
is on the Nevada side of the Oregon border, South Padre Island is a barrier
island off Texas.

They fell out because our boundary file is **simplified** — coastlines smoothed
inland, barrier islands rounded off — to keep it at 344 KB.

Week 9 said generalized boundaries are fine for coloring and wrong for
measuring. Add a third use they are wrong for: **joining**. And the join says
nothing; with `how="inner"` these three would simply never have existed.

## Part 3 — Sites and stalls per state

In [ ]:
acs = pd.read_csv("data/state_stats.csv", dtype={"fips": str})
per_state = (tagged.dropna(subset=["fips"])
             .groupby(["fips", "state_polygon"])
             .agg(sites=("name", "size"), stalls=("stalls", "sum"))
             .reset_index()
             .merge(acs[["fips", "pop_total"]], on="fips"))
per_state["per_million"] = (1e6 * per_state.sites / per_state.pop_total).round(1)

shapes = states.merge(per_state, on="fips")
conus_shapes = shapes[~shapes.state.isin(["Alaska", "Hawaii", "Puerto Rico"])]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
conus_shapes.plot(column="sites", cmap="OrRd", legend=True, ax=axes[0],
                  edgecolor="white", linewidth=0.4)
axes[0].set_title("sites per state")
conus_shapes.plot(column="stalls", cmap="OrRd", legend=True, ax=axes[1],
                  edgecolor="white", linewidth=0.4)
axes[1].set_title("stalls per state")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
print("most sites:")
print(per_state.nlargest(5, "sites")[["state_polygon", "sites", "stalls", "per_million"]]
      .to_string(index=False))
print("\nmost sites per million people:")
print(per_state.nlargest(5, "per_million")[["state_polygon", "sites", "stalls", "per_million"]]
      .to_string(index=False))

**Stalls is the better capacity measure** — a forty-stall site is not one
site's worth of charging — and the two maps differ.

And Week 9's lesson from the other side: California has by far the most and is
twelfth per person. Delaware, Wyoming and Montana lead per capita on about
twenty sites each.

Two joins of two kinds had to happen to ask this: points to polygons **by
location**, then polygons to population **by key**.

## Part 4 — The layout

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
states.plot(ax=axes[0], color="#EAF2F6", edgecolor="#2E6E8E", linewidth=0.4)
axes[0].set_title("all 52, at their real coordinates")
states[~states.state.isin(["Alaska", "Hawaii", "Puerto Rico"])].plot(
    ax=axes[1], color="#EAF2F6", edgecolor="#2E6E8E", linewidth=0.4)
axes[1].set_title("the lower 48 and DC")
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.set_aspect(1.3)
plt.tight_layout(); plt.show()

b = states.total_bounds
ak = states[states.state == "Alaska"].geometry.iloc[0]
print(f"all 52 span {b[2] - b[0]:.1f} degrees of longitude")
print(f"Alaska is {len(list(ak.geoms))} polygons; "
      f"{sum(1 for p in ak.geoms if p.bounds[0] > 0)} sit past +180, "
      f"where the Aleutians cross the antimeridian")

The **Albers USA** composite projection is the standard answer: project the
lower 48 normally, then shrink Alaska, move it, move Hawaii, and park both
below California. Plotly does it with `scope="usa"`.

In [ ]:
import plotly.express as px

abbr = tagged.dropna(subset=["fips"]).groupby("state_polygon")["state"].first()
per_state["abbr"] = per_state.state_polygon.map(abbr)

fig = px.choropleth(
    per_state, locations="abbr", locationmode="USA-states", scope="usa",
    color="per_million", color_continuous_scale="OrRd",
    hover_name="state_polygon",
    hover_data={"sites": True, "stalls": True, "per_million": ":.1f", "abbr": False},
    labels={"per_million": "sites per million"},
)
fig.update_layout(margin=dict(l=0, r=0, t=10, b=0), height=430)
fig.show()

**Alaska and Hawaii are in the bottom-left corner**, and nobody reads that as a
claim about where Alaska is. The map is deliberately wrong because the honest
version — the left panel above — is unreadable.

Hover a state and the numbers appear. That is what interactivity buys; it does
not buy honesty. The classification, the projection and the omissions are still
choices.

## Part 5 — Altair: one view drives another

In [ ]:
import altair as alt
from sklearn.neighbors import BallTree

EARTH_KM = 6371.0088
tree = BallTree(np.radians(conus[["lat", "lon"]].values), metric="haversine")
dist, idx = tree.query(np.radians(conus[["lat", "lon"]].values), k=2)

links = pd.DataFrame({
    "lat": conus.lat.values, "lon": conus.lon.values,
    "lat2": conus.lat.values[idx[:, 1]], "lon2": conus.lon.values[idx[:, 1]],
    "state": conus.state.values, "name": conus.name.values,
    "nearest_km": (dist[:, 1] * EARTH_KM).round(1),
})

pick = alt.selection_point(fields=["state"], on="mouseover", empty=False)

base = alt.Chart(states[~states.state.isin(["Alaska", "Hawaii", "Puerto Rico"])]
                 ).mark_geoshape(fill="#EAF2F6", stroke="white", strokeWidth=0.6)
edges = alt.Chart(links).mark_rule(color="#d9534f", strokeWidth=1).encode(
    longitude="lon:Q", latitude="lat:Q", longitude2="lon2:Q", latitude2="lat2:Q",
    opacity=alt.condition(pick, alt.value(0.8), alt.value(0.04)))
dots = alt.Chart(links).mark_circle(color="#2E6E8E").encode(
    longitude="lon:Q", latitude="lat:Q",
    size=alt.condition(pick, alt.value(40), alt.value(6)),
    tooltip=["name:N", "state:N", "nearest_km:Q"],
).add_params(pick)

((base + edges + dots).project(type="albersUsa")
 .properties(width=720, height=420, title="Hover a state to light up its sites"))

## Part 6 — Distance

A choropleth of sites-per-state cannot tell you how far you would have to
drive. That needs the points, and it needs the right distance.

Latitude and longitude are **angles**. A degree of latitude is always about
111 km; a degree of longitude shrinks toward the poles.

In [ ]:
for lat in (26, 38, 48):
    print(f"at {lat}N, one degree of longitude = {111.32 * np.cos(np.radians(lat)):6.1f} km")

In [ ]:
from sklearn.neighbors import BallTree

EARTH_KM = 6371.0088
# BallTree with metric="haversine" measures across the surface of a sphere.
# It wants RADIANS, and it returns an ANGLE.
tree = BallTree(np.radians(conus[["lat", "lon"]].values), metric="haversine")
dist, idx = tree.query(np.radians(conus[["lat", "lon"]].values), k=2)  # k=2: self, then nearest
conus["nearest_km"] = dist[:, 1] * EARTH_KM
conus["nearest_name"] = conus.iloc[idx[:, 1]]["name"].values

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.hist(conus.nearest_km, bins=60, color="#2E6E8E")
ax.set_xlabel("km to the nearest other Supercharger"); ax.set_ylabel("sites")
plt.tight_layout(); plt.show()

print(f"median {conus.nearest_km.median():.1f} km | mean {conus.nearest_km.mean():.1f} km"
      f" | 90th {conus.nearest_km.quantile(.9):.1f} km")

The mean is more than twice the median — the long right tail as a number.
**Report the median.**

Now look at both ends of that column.

In [ ]:
print("farthest from anything:")
print(conus.nlargest(4, "nearest_km")[["name", "state", "nearest_km"]].round(0).to_string(index=False))
print("\nclosest together:")
print(conus.nsmallest(4, "nearest_km")[["name", "state", "nearest_km"]].round(3).to_string(index=False))

One end is coverage. Hines, Oregon is 168 km from anything.

The other end is **not geography**. Thirty-six metres, and the names say it:
`Baker, CA` and `Baker (Expansion), CA`. The same place entered twice. A query
written for coverage has found a data quality problem that every map so far
hid, because at national zoom the two dots are one dot.

Decide what to do with them before counting anything.

## Part 7 — DBSCAN

Groups points that sit in dense bunches, labels the rest **noise**, and is not
told how many groups to find. Two parameters: `eps`, how far apart still counts
as close, and `min_samples`, how many neighbors anchor a group.

The geographic trap is the one from Part 6. `eps` in degrees means one distance
north-south and a different one east-west.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

EPS_KM, MIN_SAMPLES = 50, 5
coords = conus[["lat", "lon"]].values

# right: haversine on radians, eps as an angle
right = DBSCAN(eps=EPS_KM / EARTH_KM, min_samples=MIN_SAMPLES,
               metric="haversine").fit_predict(np.radians(coords))
# tempting and wrong: euclidean on raw degrees
wrong = DBSCAN(eps=EPS_KM / 111.0, min_samples=MIN_SAMPLES).fit_predict(coords)
conus["cluster"] = right

for nm, lab in [("haversine on radians", right), ("euclidean on degrees", wrong)]:
    print(f"{nm:22} {len(set(lab)) - (1 if -1 in lab else 0):3} clusters, "
          f"{(lab == -1).sum():4} noise")

Two clusters apart and a hundred noise points. But cluster *numbers* are
arbitrary — DBSCAN hands them out in discovery order — so comparing label 7 to
label 7 means nothing. Two things that do survive relabeling:

In [ ]:
print(f"adjusted Rand index: {adjusted_rand_score(right, wrong):.3f}"
      "   (1.0 would be the same grouping)")

flipped = (right == -1) != (wrong == -1)
print(f"\npoints whose noise status differs: {flipped.sum()} of {len(conus)}")
print(f"  clustered by haversine, noise under euclidean: {((right != -1) & (wrong == -1)).sum()}")
print(f"  the reverse:                                   {((right == -1) & (wrong != -1)).sum()}")

print("\nand it depends on latitude:")
for lo, hi, nm in [(24, 34, "south of 34N"), (34, 42, "34 to 42N"), (42, 50, "north of 42N")]:
    band = conus.lat.between(lo, hi).values
    print(f"  {nm:14} {band.sum():4} sites, differs for {flipped[band].mean():5.1%}")

**Narrate this.** Every disagreement runs one way: haversine finds a cluster,
Euclidean calls it noise, never the reverse. Euclidean believes a degree of
longitude is 111 km; at the Canadian border it is 74. Two stations it thinks
are 50 km apart are really 33 km apart — close enough to group, and it misses
them. The further north, the more it misses.

Nothing errors. The northern states just quietly get fewer clusters.

In [ ]:
big = (conus[conus.cluster != -1].groupby("cluster")
       .agg(sites=("name", "size"), stalls=("stalls", "sum"),
            lat=("lat", "mean"), lon=("lon", "mean"))
       .sort_values("sites", ascending=False))
print(f"{len(big)} clusters, {(conus.cluster == -1).sum()} noise. The six largest:")
print(big.head(6).round(1).to_string())

**Check them before believing them.** DBSCAN was given coordinates and nothing
else — no city names, no boundaries. Those centroids are New York and northern
New Jersey, Los Angeles, the Bay Area, central Florida, Chicago and the
Carolinas.

Recovering places you already recognize is the evidence the parameters were
sensible. The algorithm returns groups whether or not there are any.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
noise = conus[conus.cluster == -1]
ax.scatter(noise.lon, noise.lat, s=11, color="#bbbbbb", label=f"noise ({len(noise)})")
for cid in big.head(8).index:
    g = conus[conus.cluster == cid]
    ax.scatter(g.lon, g.lat, s=20, label=f"{cid} ({len(g)})")
ax.set_aspect(1.3); ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=7, loc="lower left", ncol=3)
ax.set_title(f"haversine, eps={EPS_KM} km, min_samples={MIN_SAMPLES}")
plt.tight_layout(); plt.show()

The grey points are not failures. For a charging network they may be the
**most** interesting set: the lone stations on long highways, built so a gap is
covered rather than because demand was dense.

## Part 8 — Density and the third dimension

A heat map smooths the points into a surface. The static equivalent is a
hex-binned count, and it makes the same point the class breaks made in Week 9.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, g in zip(axes, [20, 60, 150]):
    ax.hexbin(conus.lon, conus.lat, gridsize=g, cmap="inferno", mincnt=1)
    ax.set_title(f"gridsize = {g}")
    ax.set_aspect(1.3); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("the same points at three resolutions")
plt.tight_layout(); plt.show()

# the interactive version, which needs internet for the tiles:
# from folium.plugins import HeatMap
# HeatMap(conus[["lat", "lon"]].values, radius=12).add_to(m)

**The radius is the claim.** Coarse says "the coasts", fine says "individual
cities", and both are the same 2,254 points.

Now give the count a third dimension. `pydeck` wraps **deck.gl**, which is what
`st.pydeck_chart` uses, so this is the same code a Streamlit app would run.

In [ ]:
import pydeck as pdk
from IPython.display import HTML

layer = pdk.Layer(
    "HexagonLayer",
    data=conus[["lon", "lat"]],
    get_position="[lon, lat]",
    radius=35000,          # hexagon size in METRES: the binning decision
    elevation_scale=250,   # how tall a count of 1 becomes. No readable units.
    extruded=True,
    coverage=0.9,
    pickable=True,
)
view = pdk.ViewState(latitude=39, longitude=-98, zoom=3.4,
                     pitch=50,   # camera tilt. At 0 this is a flat hexbin.
                     bearing=0)

deck = pdk.Deck(layers=[layer], initial_view_state=view, map_style="light")
# map_style="light" uses a Carto basemap, which needs no Mapbox token.
HTML(deck.to_html(as_string=True))

**Drag to rotate, and watch what happens to the short columns.**

Three costs worth naming while it is on screen:

- tall columns **hide** the short ones behind them
- **perspective** makes equal values look unequal — a column at the back draws
  smaller than the same column at the front
- the basemap underneath becomes hard to read

Set `pitch=0` and it becomes a flat hex-bin map, which is more readable and
less impressive. That trade is the decision.

### The pattern worth copying

Streamlit's NYC ridesharing demo is the clearest use of this. Its shape:

1. load once with `@st.cache_data`, then **filter** on a control — an
   hour-of-day slider
2. the filtered frame feeds the `HexagonLayer`, so the slider rebuilds the map
3. the **same layer** drawn four times at different viewports: all of New York,
   then LaGuardia, JFK and Newark at `zoom=12`
4. an Altair histogram underneath, driven by the same filter

Steps 3 and 4 are the lesson. Four small maps of one dataset is a
**small-multiple**, answering "where, specifically" without making anyone pan
around. The coordinated chart answers what the map cannot: how much, exactly.

## Wrap-up → Block 3

**interact → join → aggregate → project → link → measure → group → extrude.**
Every step took a parameter: the predicate, the boundary resolution, the
classification, the projection, `eps`, `min_samples`, the hexagon radius and
the elevation scale. Block 3 runs a short version and asks you to name them.